# Analiza wyników HITL

Notebook interpretuje wyniki mini-projektu dla manipulatora przemysłowego w hali produkcyjnej. Nacisk jest położony na konsekwencje prawne i etyczne, a nie na samą optymalizację trajektorii.

In [ ]:
from pathlib import Path
import csv
from IPython.display import SVG, display, Markdown

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
WYNIKI = ROOT / "wyniki"

def czytaj_csv(nazwa):
    with (WYNIKI / nazwa).open(encoding="utf-8") as plik:
        return list(csv.DictReader(plik))

podsumowanie = czytaj_csv("podsumowanie_polityk.csv")
rekomendacje = czytaj_csv("rekomendacje_polityk.csv")
obciazenie = czytaj_csv("analiza_obciazenia_operatora.csv")
zgodnosc = czytaj_csv("macierz_zgodnosci_ai_act.csv")

## 1. Porównanie polityk

Tabela pokazuje kompromis między ryzykiem resztkowym, czasem decyzji i częstością interwencji operatora. Najniższe ryzyko nie oznacza automatycznie najlepszej polityki organizacyjnej.

In [ ]:
for w in podsumowanie:
    print(
        f"{w['polityka']}: ryzyko={w['srednie_ryzyko_resztkowe']}, "
        f"p95={w['p95_ryzyka_resztkowego']}, "
        f"czas={w['sredni_czas_decyzji_s']} s, "
        f"interwencje={float(w['odsetek_interwencji']) * 100:.2f}%"
    )

In [ ]:
display(SVG(filename=str(WYNIKI / "ryzyko_resztkowe_polityki.svg")))
display(SVG(filename=str(WYNIKI / "kompromis_ryzyko_czas.svg")))

Interpretacja: zatwierdzenie przed ruchem minimalizuje średnie ryzyko, ale wymaga częstych interwencji. Adaptacyjny HITL jest bardziej praktyczny, gdy system ma pracować powtarzalnie i nie przeciążać operatora. Prawo weta ma sens przy niższym ryzyku i znanych trajektoriach.

## 2. Scenariusze wysokiego ryzyka

Heatmapa pokazuje, że środowisko i okluzja silnie wpływają na sens nadzoru człowieka. Dla manipulatora w częściowo odseparowanej strefie ryzyko jest inne niż dla pracy bezpośrednio obok ludzi.

In [ ]:
display(SVG(filename=str(WYNIKI / "heatmapa_wysokiego_ryzyka.svg")))

Jeżeli manipulator pracuje tylko w częściowo odseparowanej hali, podstawową szkodą może być uszkodzenie mienia lub przestój. Jeżeli jednak trajektorie mogą doprowadzić do kontaktu z człowiekiem, rośnie znaczenie AI Act, rozporządzenia maszynowego i norm robotycznych. Wtedy HITL musi być tylko jednym z elementów bezpieczeństwa, obok osłon, stref, ograniczeń prędkości i awaryjnego zatrzymania.

## 3. Kiedy która polityka jest najlepsza

Najważniejszy wynik projektu jest sytuacyjny: nie należy wybierać jednej polityki dla wszystkich trajektorii.

In [ ]:
for r in rekomendacje:
    display(Markdown(
        f"**{r['sytuacja']}**  \n"
        f"Rekomendacja: `{r['rekomendowana_polityka']}`.  \n"
        f"Uzasadnienie: {r['uzasadnienie']}  \n"
        f"Warunek brzegowy: {r['warunek_brzegowy']}"
    ))

## 4. Ryzyko pozornego nadzoru

AI Act art. 14 wymaga skutecznego nadzoru człowieka. To oznacza, że operator musi mieć realną możliwość zrozumienia, zatrzymania lub poprawienia działania systemu. Sama obecność człowieka w procedurze nie wystarcza.

In [ ]:
for w in obciazenie:
    if w["model_nadzoru"] == "jeden_operator_wiele_robotow":
        print(
            f"{w['polityka']}: przeciążenie={w['indeks_przeciazenia_operatora']}, "
            f"czas={w['skorygowany_czas_decyzji_s']} s, {w['interpretacja']}"
        )

Wniosek etyczny: jeżeli jeden operator nadzoruje wiele robotów, częste eskalacje mogą stworzyć nadzór pozorny. Organizacja nie może wtedy przerzucić odpowiedzialności wyłącznie na operatora, bo to dostawca, integrator i wdrażający projektują warunki, w których operator podejmuje decyzję.

## 5. AI Act i regulacje maszynowe

Macierz zgodności pokazuje, że projekt dotyka przede wszystkim zarządzania ryzykiem, dokumentacji, logowania, przejrzystości, nadzoru człowieka oraz odporności systemu. Dla pracy blisko ludzi dochodzi silniejsze znaczenie rozporządzenia maszynowego 2023/1230 i norm robotycznych.

In [ ]:
for z in zgodnosc:
    display(Markdown(
        f"**{z['obszar']}**  \n"
        f"Wymaganie: {z['wymaganie']}  \n"
        f"W projekcie: {z['implementacja_w_projekcie']}  \n"
        f"Luka: {z['luka']}"
    ))

## 6. Wniosek końcowy

Dla manipulatora przemysłowego w częściowo odseparowanej hali najbardziej sensowny jest dobór polityki HITL zależny od kontekstu. Niskie ryzyko i znane trajektorie mogą uzasadniać prawo weta lub HITL informacyjny. Wysoka okluzja, niska pewność AI i nietypowe zadania uzasadniają adaptacyjny HITL lub zatwierdzenie przed ruchem. Jeżeli pojawia się realne ryzyko kolizji z człowiekiem, problem przestaje być tylko optymalizacją produkcyjną i wymaga pełniejszej analizy zgodności, bezpieczeństwa maszyny i skuteczności nadzoru człowieka.